In [2]:
import numpy as np
import pandas as pd
import os

from scipy.interpolate import interp1d

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

import tensorflow as tf
from tensorflow.keras import layers, Model

In [3]:
ESP32_DATA_PATH = "/Users/aleynagulkazdal/Desktop/esp32_data"

TARGET_FRAMES = 256
TARGET_SC = 90

WINDOW_SIZE = 128
STRIDE = 32

In [4]:
ACTIVITIES = [
    "still",
    "walk",
    "hand_clap",
    "horizontal_arm_wave",
    "two_hands_wave"
]

ACTIVITY_MAP = {
    act:i
    for i, act in enumerate(ACTIVITIES)
}

print(ACTIVITY_MAP)

{'still': 0, 'walk': 1, 'hand_clap': 2, 'horizontal_arm_wave': 3, 'two_hands_wave': 4}


In [5]:
def parse_esp32_csv(filepath):

    rows = []

    with open(filepath, 'r') as f:
        lines = f.readlines()

    header = lines[0].strip().split(',')
    expected_cols = len(header)

    for line in lines[1:]:

        vals = line.strip().split(',')

        if len(vals) == expected_cols:
            try:
                rows.append([float(v) for v in vals])
            except:
                continue

    if len(rows) == 0:
        raise ValueError("Hiç veri okunamadı")

    raw = np.array(rows)

    raw = raw[:, 1:]

    raw = raw[:, 4:]

    real = raw[:, 0::2]
    imag = raw[:, 1::2]

    min_sc = min(real.shape[1], imag.shape[1])

    real = real[:, :min_sc]
    imag = imag[:, :min_sc]

    amp = np.sqrt(real**2 + imag**2)

    x_old = np.linspace(0, 1, amp.shape[1])
    x_new = np.linspace(0, 1, TARGET_SC)

    interpolated = np.zeros((amp.shape[0], TARGET_SC))

    for i in range(amp.shape[0]):
        f = interp1d(x_old, amp[i], kind='linear')
        interpolated[i] = f(x_new)

    return interpolated

In [6]:
def fix_length(data, target_len=TARGET_FRAMES):

    current_len = data.shape[0]

    if current_len == target_len:
        return data

    x_old = np.linspace(0, 1, current_len)
    x_new = np.linspace(0, 1, target_len)

    f = interp1d(
        x_old,
        data,
        axis=0,
        kind='linear',
        fill_value="extrapolate"
    )

    return f(x_new)

In [7]:
def create_windows(data, window_size=WINDOW_SIZE, stride=STRIDE):

    windows = []

    for start in range(
        0,
        len(data) - window_size + 1,
        stride
    ):

        end = start + window_size

        windows.append(data[start:end])

    return np.array(windows)

In [8]:
all_files = []

for user in os.listdir(ESP32_DATA_PATH):

    user_path = os.path.join(
        ESP32_DATA_PATH,
        user
    )

    if not os.path.isdir(user_path):
        continue

    for activity, activity_label in ACTIVITY_MAP.items():

        activity_path = os.path.join(
            user_path,
            activity
        )

        if not os.path.exists(activity_path):
            continue

        csv_files = sorted([
            f for f in os.listdir(activity_path)
            if f.endswith(".csv")
        ])

        for csv_file in csv_files:

            filepath = os.path.join(
                activity_path,
                csv_file
            )

            all_files.append(
                (
                    filepath,
                    activity_label
                )
            )

print("Toplam CSV:", len(all_files))

Toplam CSV: 400


In [9]:
paths = [x[0] for x in all_files]
labels = [x[1] for x in all_files]

train_paths, test_paths, train_labels, test_labels = train_test_split(
    paths,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Train CSV:", len(train_paths))
print("Test CSV:", len(test_paths))

Train CSV: 320
Test CSV: 80


In [10]:
X_train = []
y_train = []

for filepath, label in zip(
    train_paths,
    train_labels
):

    try:

        data = parse_esp32_csv(filepath)

        data = fix_length(data)

        windows = create_windows(data)

        for w in windows:

            X_train.append(w)
            y_train.append(label)

    except Exception as e:

        print("HATA:", filepath)
        print(e)

X_train = np.array(X_train)
y_train = np.array(y_train)

print(X_train.shape)
print(y_train.shape)

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/scipy/interpolate/_interpolate.py:479: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]


(1600, 128, 90)
(1600,)


In [11]:
X_test = []
y_test = []

for filepath, label in zip(
    test_paths,
    test_labels
):

    try:

        data = parse_esp32_csv(filepath)

        data = fix_length(data)

        windows = create_windows(data)

        for w in windows:

            X_test.append(w)
            y_test.append(label)

    except Exception as e:

        print("HATA:", filepath)
        print(e)

X_test = np.array(X_test)
y_test = np.array(y_test)

print(X_test.shape)
print(y_test.shape)

(400, 128, 90)
(400,)


In [12]:
X_train_norm = (
    X_train -
    X_train.mean(
        axis=(1,2),
        keepdims=True
    )
) / (
    X_train.std(
        axis=(1,2),
        keepdims=True
    ) + 1e-8
)

print("Train normalize tamam ")

Train normalize tamam 


In [13]:
X_test_norm = (
    X_test -
    X_test.mean(
        axis=(1,2),
        keepdims=True
    )
) / (
    X_test.std(
        axis=(1,2),
        keepdims=True
    ) + 1e-8
)

print("Test normalize tamam ")

Test normalize tamam 


In [14]:
train_nan_mask = np.isnan(
    X_train_norm
).any(axis=(1,2))

test_nan_mask = np.isnan(
    X_test_norm
).any(axis=(1,2))

print(
    "NaN train sample:",
    train_nan_mask.sum()
)

print(
    "NaN test sample:",
    test_nan_mask.sum()
)

NaN train sample: 20
NaN test sample: 5


In [15]:
X_train_clean = X_train_norm[
    ~train_nan_mask
]

y_train_clean = y_train[
    ~train_nan_mask
]

X_test_clean = X_test_norm[
    ~test_nan_mask
]

y_test_clean = y_test[
    ~test_nan_mask
]

print(X_train_clean.shape)
print(X_test_clean.shape)

(1580, 128, 90)
(395, 128, 90)


In [16]:
print(np.isnan(X_train_clean).sum())
print(np.isnan(X_test_clean).sum())

0
0


In [17]:
inputs = layers.Input(shape=(128,90))

x = layers.Conv1D(
    32,
    kernel_size=5,
    activation='relu',
    padding='same'
)(inputs)

x = layers.BatchNormalization()(x)

x = layers.MaxPooling1D(2)(x)

x = layers.Dropout(0.3)(x)

x = layers.Conv1D(
    64,
    kernel_size=3,
    activation='relu',
    padding='same'
)(x)

x = layers.BatchNormalization()(x)

x = layers.MaxPooling1D(2)(x)

x = layers.Dropout(0.3)(x)

x = layers.GlobalAveragePooling1D()(x)

x = layers.Dense(
    64,
    activation='relu'
)(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    len(ACTIVITIES),
    activation='softmax'
)(x)

activity_model = Model(inputs, outputs)

activity_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 90)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 128, 32)        │        14,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 64, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 64, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 32, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,509 (99.64 KB)

 Trainable params: 25,317 (98.89 KB)

 Non-trainable params: 192 (768.00 B)

In [18]:
activity_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [19]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

In [20]:
history = activity_model.fit(
    X_train_clean,
    y_train_clean,
    validation_data=(
        X_test_clean,
        y_test_clean
    ),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.4627 - loss: 1.3173 - val_accuracy: 0.3013 - val_loss: 1.4739
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6367 - loss: 0.9600 - val_accuracy: 0.4557 - val_loss: 1.3614
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7013 - loss: 0.7908 - val_accuracy: 0.6127 - val_loss: 1.1793
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7620 - loss: 0.6416 - val_accuracy: 0.5114 - val_loss: 1.4044
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7873 - loss: 0.5716 - val_accuracy: 0.5899 - val_loss: 1.2665
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8259 - loss: 0.4768 - val_accuracy: 0.7114 - val_loss: 1.0827
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8443 - loss: 0.4539 - val_accuracy: 0.5494 - val_loss: 1.4602
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8563 - loss: 0.3999 - val_accuracy: 0.5215 - val_los

In [21]:
y_pred_probs = activity_model.predict(
    X_test_clean
)

y_pred = np.argmax(
    y_pred_probs,
    axis=1
)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


In [22]:
cm = confusion_matrix(
    y_test_clean,
    y_pred
)

print(cm)

[[55 10  0 10  5]
 [ 6 39 12  9 14]
 [ 5 10 50  0 15]
 [ 0  0  4 66  5]
 [ 1  0  0  8 71]]


In [23]:
print(
    classification_report(
        y_test_clean,
        y_pred,
        target_names=ACTIVITIES
    )
)

                     precision    recall  f1-score   support

              still       0.82      0.69      0.75        80
               walk       0.66      0.49      0.56        80
          hand_clap       0.76      0.62      0.68        80
horizontal_arm_wave       0.71      0.88      0.79        75
     two_hands_wave       0.65      0.89      0.75        80

           accuracy                           0.71       395
          macro avg       0.72      0.71      0.71       395
       weighted avg       0.72      0.71      0.70       395



In [24]:
np.save("X_train_clean.npy", X_train_clean)
np.save("y_train_clean.npy", y_train_clean)

np.save("X_test_clean.npy", X_test_clean)
np.save("y_test_clean.npy", y_test_clean)

print("ESP32 dataset kaydedildi.")

ESP32 dataset kaydedildi.
